In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from tqdm import tqdm
import matplotlib.pyplot as plt

import timm

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"device: {device}")

device: cpu


In [2]:
BATCH_SIZE = 16  
EPOCHS = 15      
LEARNING_RATE = 1e-3
IMG_SIZE = 300   

In [3]:
# path
TRAIN_PATH = '../data/cropped'

# transform
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15), 
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# set dataset
full_dataset = datasets.ImageFolder(
    root=TRAIN_PATH,
    transform=val_transform  # default val dulu
)

class_names = full_dataset.classes

# split 80:20
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=generator
)

train_dataset.dataset.transform = train_transform
val_dataset.dataset.transform = val_transform

# dataloader
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=2)

print(f"Data --> Train: {len(train_dataset)}, Val: {len(val_dataset)}")

Data --> Train: 1176, Val: 295


In [5]:
model = timm.create_model('convnext_tiny', pretrained=True, num_classes=len(class_names), drop_rate=0.3)
model = model.to(device)

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss()
scaler = torch.cuda.amp.GradScaler()

model_dir = '../Models/' 
model_save_path = os.path.join(model_dir, 'convnext_tiny_model.pth')

print("Model ConvNeXt Tiny siap dilatih!")

Model ConvNeXt Tiny siap dilatih!


c:\Users\nalin\anaconda3\envs\coba2env\Lib\site-packages\torch\cuda\amp\grad_scaler.py:126: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


In [6]:
train_losses, val_losses, val_accuracies = [], [], []
best_val_acc = 0.0

patience = 3 
patience_counter = 0

print("Memulai proses Training ConvNeXt...")
for epoch in range(EPOCHS):
    
    model.train()
    running_loss = 0.0
    
    train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]')
    for inputs, labels in train_pbar:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        
        with torch.amp.autocast('cuda'):
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * inputs.size(0)
        current_lr = scheduler.get_last_lr()[0]
        train_pbar.set_postfix({'loss': f"{loss.item():.4f}", 'lr': f"{current_lr:.6f}"})
        
    scheduler.step()
    epoch_train_loss = running_loss / len(train_dataset)
    train_losses.append(epoch_train_loss)
    
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        val_pbar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Val]')
        for inputs, labels in val_pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    epoch_val_loss = val_loss / len(val_dataset)
    epoch_val_acc = correct / total
    
    val_losses.append(epoch_val_loss)
    val_accuracies.append(epoch_val_acc)
    
    print(f"Epoch {epoch+1} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")
    
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        
        os.makedirs(model_dir, exist_ok=True)
        torch.save(model.state_dict(), model_save_path)
        
        print(f"Model terbaik baru disimpan di {model_save_path}! (Akurasi: {best_val_acc:.4f})")
        patience_counter = 0 
    else:
        patience_counter += 1
        print(f"Tidak ada perbaikan. Early stopping counter: {patience_counter}/{patience}")
        
    if patience_counter >= patience:
        print(f"\nEarly Stopping aktif! Proses dihentikan pada Epoch {epoch+1}.")
        break

Memulai proses Training ConvNeXt...


Epoch 1/15 [Train]:   0%|          | 0/74 [00:00<?, ?it/s]c:\Users\nalin\anaconda3\envs\coba2env\Lib\site-packages\torch\amp\autocast_mode.py:250: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


: 